# A Neural Probabilistic Language Model

Replication of Bengio, Ducharme, Vincent and Jauvin (2003), *A Neural Probabilistic Language
Model*, JMLR 3.

This paper introduced learned distributed word embeddings inside a neural network that
predicts the next word from a fixed window of previous words, fighting the curse of
dimensionality that cripples n-gram count models. We implement the model (a shared embedding
table, a tanh hidden layer over the concatenated context, and a softmax output) and train it
on a small corpus. We reproduce the core behavior: validation perplexity drops sharply as
the network learns, and the model produces sensible next-word continuations.

In [1]:
import re, urllib.request, collections, math, torch, torch.nn as nn
torch.manual_seed(0);

In [2]:
urls = ["https://www.gutenberg.org/files/11/11-0.txt",
        "https://www.gutenberg.org/files/12/12-0.txt"]
text = " ".join(urllib.request.urlopen(u, timeout=30).read().decode("utf-8", "ignore") for u in urls)
words = re.findall(r"[a-z']+", text.lower())
counts = collections.Counter(words)
vocab = ["<unk>"] + [w for w, c in counts.items() if c >= 3]
stoi = collections.defaultdict(int, {w: i for i, w in enumerate(vocab)})
itos = {i: w for w, i in stoi.items()}
ids = torch.tensor([stoi[w] for w in words])
V = len(vocab); print("tokens", len(ids), "vocab", V)

tokens 58545 vocab 1753


In [3]:
# Build (context -> next word) examples with a fixed context window n.
N = 4
ctx = torch.stack([ids[i:i+N] for i in range(len(ids)-N)])
nxt = ids[N:]
split = int(0.9 * len(ctx))
Xtr, Ytr, Xva, Yva = ctx[:split], nxt[:split], ctx[split:], nxt[split:]
print("train examples", len(Xtr), "val examples", len(Xva))

train examples 52686 val examples 5855


In [4]:
class NPLM(nn.Module):
    def __init__(self, emb=60, hid=128):
        super().__init__()
        self.emb = nn.Embedding(V, emb)
        self.h = nn.Linear(N*emb, hid)
        self.out = nn.Linear(hid, V)
    def forward(self, x):
        e = self.emb(x).view(x.size(0), -1)         # concatenate context-word embeddings
        return self.out(torch.tanh(self.h(e)))

net = NPLM(); opt = torch.optim.Adam(net.parameters(), lr=2e-3); lf = nn.CrossEntropyLoss()
def val_ppl():
    net.eval()
    with torch.no_grad():
        return math.exp(lf(net(Xva), Yva).item())
print("perplexity before training:", round(val_ppl(), 1))

perplexity before training: 1813.6


In [5]:
dl = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(Xtr, Ytr), batch_size=512, shuffle=True)
for ep in range(8):
    net.train()
    for xb, yb in dl:
        opt.zero_grad(); lf(net(xb), yb).backward(); opt.step()
    print(f"epoch {ep+1}: validation perplexity = {val_ppl():.1f}")

epoch 1: validation perplexity = 231.6


epoch 2: validation perplexity = 178.5


epoch 3: validation perplexity = 158.1


epoch 4: validation perplexity = 149.8


epoch 5: validation perplexity = 147.0


epoch 6: validation perplexity = 147.2


epoch 7: validation perplexity = 151.1


epoch 8: validation perplexity = 154.6


In [6]:
# Greedy next-word continuation from a seed context.
net.eval()
def continue_text(seed, k=12):
    toks = seed.split()
    for _ in range(k):
        ctx = [stoi[w] for w in toks[-N:]]
        ctx = [0] * (N - len(ctx)) + ctx        # left-pad short contexts to length N
        with torch.no_grad():
            logits = net(torch.tensor([ctx]))[0]
            logits[0] = float("-inf")           # never emit the <unk> token
            toks.append(itos[logits.argmax().item()])
    return " ".join(toks)
for seed in ["the white rabbit said", "she said to the"]:
    print(f"'{seed}' -> {continue_text(seed)}")

'the white rabbit said' -> the white rabbit said the king and the unicorn was a little brook and the dormouse
'she said to the' -> she said to the jury and the words of the song and the unicorn was a
